In [ ]:

from pathlib import Path
from datetime import datetime
import json
import duckdb
import pandas as pd

# ============================================================
# 1. CONFIGURATION
# ============================================================

INPUT_CSV = Path(
    r"C:\Users\Kavya m\OneDrive\Desktop\tsts3\2019-Nov.csv"
)

BASE_OUTPUT_DIR = Path(
    r"C:\Users\Kavya m\OneDrive\Desktop\tsts3"
)

# Each run gets a new folder to avoid overwriting old results.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = BASE_OUTPUT_DIR / f"ecommerce_analysis_{RUN_ID}"

CLEAN_DIR = OUTPUT_DIR / "cleaned_parquet"
TEMP_DIR = OUTPUT_DIR / "duckdb_temp"
REPORT_DIR = OUTPUT_DIR / "reports"

MEMORY_LIMIT = "3GB"
THREADS = 2

# Exact duplicate check is expensive on 67.5 million rows.
CHECK_EXACT_DUPLICATES = True

# Keep False to preserve all rows.
REMOVE_EXACT_DUPLICATES = False

EXPECTED_COLUMNS = [
    "event_time",
    "event_type",
    "product_id",
    "category_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
]

VALID_EVENT_TYPES = [
    "view",
    "cart",
    "remove_from_cart",
    "purchase",
]


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def sql_string(value):
    """Safely quote a SQL string literal."""
    return "'" + str(value).replace("'", "''") + "'"


def export_query(query, filename):
    """Execute a report query and save the result as CSV."""
    df = con.execute(query).df()
    df.to_csv(REPORT_DIR / filename, index=False)
    return df


def export_funnel_summary(dimensions, filename):
    """
    dimensions: list of (SQL expression, output alias) tuples.
    """

    select_dimensions = ", ".join(
        f"{expr} AS {alias}"
        for expr, alias in dimensions
    )

    group_by = ", ".join(
        expr for expr, _ in dimensions
    )

    prefix = select_dimensions + "," if dimensions else ""
    group_clause = f"GROUP BY {group_by}" if dimensions else ""
    order_clause = f"ORDER BY {group_by}" if dimensions else ""

    query = f"""
        SELECT
            {prefix}

            SUM(viewed) AS viewed_product_sessions,
            SUM(added_to_cart) AS cart_product_sessions,
            SUM(purchased) AS purchased_product_sessions,

            ROUND(
                100.0 * SUM(added_to_cart)
                / NULLIF(SUM(viewed), 0),
                2
            ) AS view_to_cart_rate_pct,

            ROUND(
                100.0 * SUM(purchased)
                / NULLIF(SUM(added_to_cart), 0),
                2
            ) AS cart_to_purchase_rate_pct,

            ROUND(
                100.0 * SUM(purchased)
                / NULLIF(SUM(viewed), 0),
                2
            ) AS view_to_purchase_rate_pct

        FROM product_session_funnel
        {group_clause}
        {order_clause}
    """

    return export_query(query, filename)


# ============================================================
# 3. CHECK INPUT FILE AND CREATE OUTPUT FOLDERS
# ============================================================

if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"CSV file not found: {INPUT_CSV}"
    )

for folder in [CLEAN_DIR, TEMP_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("E-COMMERCE DATA CLEANING AND FUNNEL ANALYSIS")
print("=" * 60)

print(f"Input file: {INPUT_CSV}")
print(f"Output folder: {OUTPUT_DIR}")

con = duckdb.connect(
    str(OUTPUT_DIR / "analysis.duckdb")
)

con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
con.execute(f"SET threads = {THREADS}")
con.execute(
    f"SET temp_directory = '{TEMP_DIR.as_posix()}'"
)
con.execute("SET preserve_insertion_order = false")


# ============================================================
# 4. READ CSV
# ============================================================

csv_path = sql_string(INPUT_CSV.as_posix())

con.execute(f"""
    CREATE VIEW raw_events AS
    SELECT *
    FROM read_csv(
        {csv_path},
        header = true,
        all_varchar = true,
        nullstr = ['', 'NULL', 'null', 'NA', 'N/A']
    )
""")

actual_columns = [
    row[0]
    for row in con.execute(
        "DESCRIBE raw_events"
    ).fetchall()
]

missing_columns = [
    col for col in EXPECTED_COLUMNS
    if col not in actual_columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}\n"
        f"Columns found: {actual_columns}"
    )

raw_count = con.execute(
    "SELECT COUNT(*) FROM raw_events"
).fetchone()[0]

print(f"\nRaw rows: {raw_count:,}")


# ============================================================
# 5. EXACT DUPLICATE DETECTION - CORRECTED
# ============================================================

exact_duplicates_found = None

if CHECK_EXACT_DUPLICATES:
    print("\nChecking exact duplicate rows...")
    print("This may take considerable time and disk space.")

    duplicate_result = con.execute("""
        SELECT COALESCE(SUM(row_count - 1), 0)
        FROM (
            SELECT
                event_time,
                event_type,
                product_id,
                category_id,
                category_code,
                brand,
                price,
                user_id,
                user_session,
                COUNT(*) AS row_count
            FROM raw_events
            GROUP BY
                event_time,
                event_type,
                product_id,
                category_id,
                category_code,
                brand,
                price,
                user_id,
                user_session
            HAVING COUNT(*) > 1
        ) AS duplicate_groups
    """).fetchone()

    exact_duplicates_found = int(duplicate_result[0])

    print(
        f"Exact duplicate rows found: "
        f"{exact_duplicates_found:,}"
    )

else:
    print("\nExact duplicate check skipped.")

if REMOVE_EXACT_DUPLICATES:
    con.execute("""
        CREATE VIEW source_events AS
        SELECT DISTINCT *
        FROM raw_events
    """)
else:
    con.execute("""
        CREATE VIEW source_events AS
        SELECT *
        FROM raw_events
    """)

source_count = con.execute(
    "SELECT COUNT(*) FROM source_events"
).fetchone()[0]

duplicates_removed = raw_count - source_count

print(f"Rows before deduplication: {raw_count:,}")
print(f"Rows after deduplication: {source_count:,}")
print(f"Exact duplicate rows removed: {duplicates_removed:,}")

# ============================================================
# 6. NORMALIZE MISSING VALUES
# ============================================================

con.execute("""
    CREATE VIEW normalized_events AS
    SELECT
        NULLIF(TRIM(event_time), '') AS event_time,
        NULLIF(LOWER(TRIM(event_type)), '') AS event_type,
        NULLIF(TRIM(product_id), '') AS product_id,
        NULLIF(TRIM(category_id), '') AS category_id,
        NULLIF(TRIM(category_code), '') AS category_code,
        NULLIF(TRIM(brand), '') AS brand,
        NULLIF(TRIM(price), '') AS price,
        NULLIF(TRIM(user_id), '') AS user_id,
        NULLIF(TRIM(user_session), '') AS user_session
    FROM source_events
""")


# ============================================================
# 7. CONVERT DATA TYPES AND VALIDATE VALUES
# ============================================================

con.execute("""
    CREATE VIEW typed_events AS
    SELECT
        TRY_CAST(event_time AS TIMESTAMPTZ) AS event_time,
        event_type,
        TRY_CAST(product_id AS BIGINT) AS product_id,
        TRY_CAST(category_id AS BIGINT) AS category_id,
        category_code,
        brand,
        TRY_CAST(price AS DOUBLE) AS price,
        TRY_CAST(user_id AS BIGINT) AS user_id,
        user_session
    FROM normalized_events
""")

con.execute("""
    CREATE VIEW cleaned_events AS
    SELECT
        event_time,
        event_type,
        product_id,
        category_id,
        category_code,
        brand,

        CASE
            WHEN price IS NOT NULL
             AND isfinite(price)
             AND price >= 0
            THEN price
            ELSE NULL
        END AS price,

        user_id,
        user_session,
        CAST(event_time AS DATE) AS event_date,
        DATE_TRUNC('month', event_time) AS event_month

    FROM typed_events
    WHERE event_time IS NOT NULL
      AND event_type IN (
          'view',
          'cart',
          'remove_from_cart',
          'purchase'
      )
""")


# ============================================================
# 8. DATA QUALITY REPORT
# ============================================================

clean_count = con.execute(
    "SELECT COUNT(*) FROM cleaned_events"
).fetchone()[0]

invalid_timestamp_count = con.execute("""
    SELECT COUNT(*)
    FROM typed_events
    WHERE event_time IS NULL
""").fetchone()[0]

invalid_event_type_count = con.execute("""
    SELECT COUNT(*)
    FROM typed_events
    WHERE event_type IS NULL
       OR event_type NOT IN (
           'view',
           'cart',
           'remove_from_cart',
           'purchase'
       )
""").fetchone()[0]

invalid_price_count = con.execute("""
    SELECT COUNT(*)
    FROM typed_events
    WHERE price IS NULL
       OR NOT isfinite(price)
       OR price < 0
""").fetchone()[0]

missing_rows = []

for column in EXPECTED_COLUMNS:
    count = con.execute(f"""
        SELECT COUNT(*)
        FROM normalized_events
        WHERE {column} IS NULL
    """).fetchone()[0]

    missing_rows.append({
        "column": column,
        "missing_count": count,
        "missing_percentage": round(
            100 * count / source_count, 4
        ) if source_count else 0
    })

pd.DataFrame(missing_rows).to_csv(
    REPORT_DIR / "missing_values_report.csv",
    index=False
)

quality_report = {
    "raw_rows": raw_count,
    "exact_duplicate_check_enabled": CHECK_EXACT_DUPLICATES,
    "exact_duplicate_rows_found": exact_duplicates_found,
    "exact_duplicate_removal_enabled": REMOVE_EXACT_DUPLICATES,
    "exact_duplicate_rows_removed": duplicates_removed,
    "rows_after_duplicate_handling": source_count,
    "cleaned_rows": clean_count,
    "excluded_rows": source_count - clean_count,
    "invalid_or_missing_timestamps": invalid_timestamp_count,
    "invalid_or_missing_event_types": invalid_event_type_count,
    "invalid_or_missing_prices": invalid_price_count,
}

with open(
    REPORT_DIR / "data_quality_report.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(quality_report, file, indent=4)

print("\nData quality report:")
print(json.dumps(quality_report, indent=4))


# ============================================================
# 9. EXPORT CLEANED DATA TO PARQUET
# ============================================================

clean_path = sql_string(CLEAN_DIR.as_posix())

con.execute(f"""
    COPY (
        SELECT * FROM cleaned_events
    )
    TO {clean_path}
    (
        FORMAT PARQUET,
        PARTITION_BY (event_date),
        COMPRESSION ZSTD
    )
""")

print(f"\nCleaned data saved to: {CLEAN_DIR}")


# ============================================================
# 10. READ CLEANED PARQUET FILES
# ============================================================

parquet_glob = sql_string(
    (CLEAN_DIR / "**" / "*.parquet").as_posix()
)

con.execute(f"""
    CREATE VIEW parquet_events AS
    SELECT *
    FROM read_parquet(
        {parquet_glob},
        hive_partitioning = true
    )
""")


# ============================================================
# 11. BUILD PRODUCT-SESSION FUNNEL
# ============================================================

# Funnel unit: user + session + product.
#
# View: product was viewed.
# Cart: a cart event occurred after the first view.
# Purchase: a purchase occurred after the qualifying cart.
#
# Missing user/session/product IDs are excluded from the funnel.
# remove_from_cart is retained in cleaned events but is not
# used to determine whether the cart was continuously retained.

con.execute("""
    CREATE VIEW product_session_funnel AS

    WITH valid_events AS (
        SELECT *
        FROM parquet_events
        WHERE user_id IS NOT NULL
          AND user_session IS NOT NULL
          AND product_id IS NOT NULL
    ),

    first_views AS (
        SELECT
            user_id,
            user_session,
            product_id,
            MIN(event_time) AS first_view
        FROM valid_events
        WHERE event_type = 'view'
        GROUP BY user_id, user_session, product_id
    ),

    first_carts AS (
        SELECT
            v.user_id,
            v.user_session,
            v.product_id,
            v.first_view,
            MIN(e.event_time) AS qualifying_cart
        FROM first_views v
        LEFT JOIN valid_events e
          ON e.user_id = v.user_id
         AND e.user_session = v.user_session
         AND e.product_id = v.product_id
         AND e.event_type = 'cart'
         AND e.event_time > v.first_view
        GROUP BY
            v.user_id, v.user_session, v.product_id, v.first_view
    ),

    first_purchases AS (
        SELECT
            c.user_id,
            c.user_session,
            c.product_id,
            c.first_view,
            c.qualifying_cart,
            MIN(e.event_time) AS qualifying_purchase
        FROM first_carts c
        LEFT JOIN valid_events e
          ON e.user_id = c.user_id
         AND e.user_session = c.user_session
         AND e.product_id = c.product_id
         AND e.event_type = 'purchase'
         AND c.qualifying_cart IS NOT NULL
         AND e.event_time > c.qualifying_cart
        GROUP BY
            c.user_id,
            c.user_session,
            c.product_id,
            c.first_view,
            c.qualifying_cart
    ),

    dimensions AS (
        SELECT
            user_id,
            user_session,
            product_id,
            ANY_VALUE(category_code) AS category_code,
            ANY_VALUE(brand) AS brand
        FROM valid_events
        GROUP BY user_id, user_session, product_id
    )

    SELECT
        p.user_id,
        p.user_session,
        p.product_id,
        p.first_view,
        p.qualifying_cart,
        p.qualifying_purchase,
        d.category_code,
        d.brand,
        1 AS viewed,

        CASE
            WHEN p.qualifying_cart IS NOT NULL THEN 1
            ELSE 0
        END AS added_to_cart,

        CASE
            WHEN p.qualifying_purchase IS NOT NULL THEN 1
            ELSE 0
        END AS purchased

    FROM first_purchases p
    LEFT JOIN dimensions d
      ON d.user_id = p.user_id
     AND d.user_session = p.user_session
     AND d.product_id = p.product_id
""")


# ============================================================
# 12. EXPORT FUNNEL REPORTS
# ============================================================

overall = export_funnel_summary(
    [],
    "overall_funnel.csv"
)

daily = export_funnel_summary(
    [("CAST(first_view AS DATE)", "view_date")],
    "daily_funnel.csv"
)

category = export_funnel_summary(
    [("COALESCE(category_code, 'Unknown')", "category")],
    "category_funnel.csv"
)

brand = export_funnel_summary(
    [("COALESCE(brand, 'Unknown')", "brand")],
    "brand_funnel.csv"
)


# ============================================================
# 13. EXPORT PRODUCT-SESSION FUNNEL
# ============================================================

funnel_path = sql_string(
    (OUTPUT_DIR / "product_session_funnel.parquet").as_posix()
)

con.execute(f"""
    COPY product_session_funnel
    TO {funnel_path}
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("\nOverall funnel:")
print(overall.to_string(index=False))

print("\nReports saved to:", REPORT_DIR)


# ============================================================
# 14. CLOSE DATABASE
# ============================================================

con.close()

print("\nAnalysis completed successfully.")
print("Output folder:", OUTPUT_DIR)

E-COMMERCE DATA CLEANING AND FUNNEL ANALYSIS
Input file: C:\Users\Kavya m\OneDrive\Desktop\tsts3\2019-Nov.csv
Output folder: C:\Users\Kavya m\OneDrive\Desktop\tsts3\ecommerce_analysis_20260928_144508


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:

from pathlib import Path
from datetime import datetime
import json
import duckdb
import pandas as pd

# ============================================================
# 1. CONFIGURATION
# ============================================================

INPUT_CSV = Path(
    r"C:\Users\Kavya m\OneDrive\Desktop\tsts3\2019-Nov.csv"
)

BASE_OUTPUT_DIR = Path(
    r"C:\Users\Kavya m\OneDrive\Desktop\tsts3"
)

# Each run gets a new folder to avoid overwriting old results.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = BASE_OUTPUT_DIR / f"ecommerce_analysis_{RUN_ID}"

CLEAN_DIR = OUTPUT_DIR / "cleaned_parquet"
TEMP_DIR = OUTPUT_DIR / "duckdb_temp"
REPORT_DIR = OUTPUT_DIR / "reports"

MEMORY_LIMIT = "3GB"
THREADS = 2

# Exact duplicate check is expensive on 67.5 million rows.
CHECK_EXACT_DUPLICATES = True

# Keep False to preserve all rows.
REMOVE_EXACT_DUPLICATES = True

EXPECTED_COLUMNS = [
    "event_time",
    "event_type",
    "product_id",
    "category_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
]

VALID_EVENT_TYPES = [
    "view",
    "cart",
    "remove_from_cart",
    "purchase",
]


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def sql_string(value):
    """Safely quote a SQL string literal."""
    return "'" + str(value).replace("'", "''") + "'"


def export_query(query, filename):
    """Execute a report query and save the result as CSV."""
    df = con.execute(query).df()
    df.to_csv(REPORT_DIR / filename, index=False)
    return df


def export_funnel_summary(dimensions, filename):
    """
    dimensions: list of (SQL expression, output alias) tuples.
    """

    select_dimensions = ", ".join(
        f"{expr} AS {alias}"
        for expr, alias in dimensions
    )

    group_by = ", ".join(
        expr for expr, _ in dimensions
    )

    prefix = select_dimensions + "," if dimensions else ""
    group_clause = f"GROUP BY {group_by}" if dimensions else ""
    order_clause = f"ORDER BY {group_by}" if dimensions else ""

    query = f"""
        SELECT
            {prefix}

            SUM(viewed) AS viewed_product_sessions,
            SUM(added_to_cart) AS cart_product_sessions,
            SUM(purchased) AS purchased_product_sessions,

            ROUND(
                100.0 * SUM(added_to_cart)
                / NULLIF(SUM(viewed), 0),
                2
            ) AS view_to_cart_rate_pct,

            ROUND(
                100.0 * SUM(purchased)
                / NULLIF(SUM(added_to_cart), 0),
                2
            ) AS cart_to_purchase_rate_pct,

            ROUND(
                100.0 * SUM(purchased)
                / NULLIF(SUM(viewed), 0),
                2
            ) AS view_to_purchase_rate_pct

        FROM product_session_funnel
        {group_clause}
        {order_clause}
    """

    return export_query(query, filename)


# ============================================================
# 3. CHECK INPUT FILE AND CREATE OUTPUT FOLDERS
# ============================================================

if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"CSV file not found: {INPUT_CSV}"
    )

for folder in [CLEAN_DIR, TEMP_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("E-COMMERCE DATA CLEANING AND FUNNEL ANALYSIS")
print("=" * 60)

print(f"Input file: {INPUT_CSV}")
print(f"Output folder: {OUTPUT_DIR}")

con = duckdb.connect(
    str(OUTPUT_DIR / "analysis.duckdb")
)

con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
con.execute(f"SET threads = {THREADS}")
con.execute(
    f"SET temp_directory = '{TEMP_DIR.as_posix()}'"
)
con.execute("SET preserve_insertion_order = false")


# ============================================================
# 4. READ CSV
# ============================================================

csv_path = sql_string(INPUT_CSV.as_posix())

con.execute(f"""
    CREATE VIEW raw_events AS
    SELECT *
    FROM read_csv(
        {csv_path},
        header = true,
        all_varchar = true,
        nullstr = ['', 'NULL', 'null', 'NA', 'N/A']
    )
""")

actual_columns = [
    row[0]
    for row in con.execute(
        "DESCRIBE raw_events"
    ).fetchall()
]

missing_columns = [
    col for col in EXPECTED_COLUMNS
    if col not in actual_columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}\n"
        f"Columns found: {actual_columns}"
    )

raw_count = con.execute(
    "SELECT COUNT(*) FROM raw_events"
).fetchone()[0]

print(f"\nRaw rows: {raw_count:,}")


# ============================================================
# 5. EXACT DUPLICATE DETECTION - CORRECTED
# ============================================================

exact_duplicates_found = None

if CHECK_EXACT_DUPLICATES:
    print("\nChecking exact duplicate rows...")
    print("This may take considerable time and disk space.")

    duplicate_result = con.execute("""
        SELECT COALESCE(SUM(row_count - 1), 0)
        FROM (
            SELECT
                event_time,
                event_type,
                product_id,
                category_id,
                category_code,
                brand,
                price,
                user_id,
                user_session,
                COUNT(*) AS row_count
            FROM raw_events
            GROUP BY
                event_time,
                event_type,
                product_id,
                category_id,
                category_code,
                brand,
                price,
                user_id,
                user_session
            HAVING COUNT(*) > 1
        ) AS duplicate_groups
    """).fetchone()

    exact_duplicates_found = int(duplicate_result[0])

    print(
        f"Exact duplicate rows found: "
        f"{exact_duplicates_found:,}"
    )

else:
    print("\nExact duplicate check skipped.")

if REMOVE_EXACT_DUPLICATES:
    con.execute("""
        CREATE VIEW source_events AS
        SELECT DISTINCT *
        FROM raw_events
    """)
else:
    con.execute("""
        CREATE VIEW source_events AS
        SELECT *
        FROM raw_events
    """)

source_count = con.execute(
    "SELECT COUNT(*) FROM source_events"
).fetchone()[0]

duplicates_removed = raw_count - source_count

print(f"Rows before deduplication: {raw_count:,}")
print(f"Rows after deduplication: {source_count:,}")
print(f"Exact duplicate rows removed: {duplicates_removed:,}")

# ============================================================
# 6. NORMALIZE MISSING VALUES
# ============================================================

con.execute("""
    CREATE VIEW normalized_events AS
    SELECT
        NULLIF(TRIM(event_time), '') AS event_time,
        NULLIF(LOWER(TRIM(event_type)), '') AS event_type,
        NULLIF(TRIM(product_id), '') AS product_id,
        NULLIF(TRIM(category_id), '') AS category_id,
        NULLIF(TRIM(category_code), '') AS category_code,
        NULLIF(TRIM(brand), '') AS brand,
        NULLIF(TRIM(price), '') AS price,
        NULLIF(TRIM(user_id), '') AS user_id,
        NULLIF(TRIM(user_session), '') AS user_session
    FROM source_events
""")


# ============================================================
# 7. CONVERT DATA TYPES AND VALIDATE VALUES
# ============================================================

con.execute("""
    CREATE VIEW typed_events AS
    SELECT
        TRY_CAST(event_time AS TIMESTAMPTZ) AS event_time,
        event_type,
        TRY_CAST(product_id AS BIGINT) AS product_id,
        TRY_CAST(category_id AS BIGINT) AS category_id,
        category_code,
        brand,
        TRY_CAST(price AS DOUBLE) AS price,
        TRY_CAST(user_id AS BIGINT) AS user_id,
        user_session
    FROM normalized_events
""")

con.execute("""
    CREATE VIEW cleaned_events AS
    SELECT
        event_time,
        event_type,
        product_id,
        category_id,
        category_code,
        brand,

        CASE
            WHEN price IS NOT NULL
             AND isfinite(price)
             AND price >= 0
            THEN price
            ELSE NULL
        END AS price,

        user_id,
        user_session,
        CAST(event_time AS DATE) AS event_date,
        DATE_TRUNC('month', event_time) AS event_month

    FROM typed_events
    WHERE event_time IS NOT NULL
      AND event_type IN (
          'view',
          'cart',
          'remove_from_cart',
          'purchase'
      )
""")


# ============================================================
# 8. DATA QUALITY REPORT
# ============================================================

clean_count = con.execute(
    "SELECT COUNT(*) FROM cleaned_events"
).fetchone()[0]

invalid_timestamp_count = con.execute("""
    SELECT COUNT(*)
    FROM typed_events
    WHERE event_time IS NULL
""").fetchone()[0]

invalid_event_type_count = con.execute("""
    SELECT COUNT(*)
    FROM typed_events
    WHERE event_type IS NULL
       OR event_type NOT IN (
           'view',
           'cart',
           'remove_from_cart',
           'purchase'
       )
""").fetchone()[0]

invalid_price_count = con.execute("""
    SELECT COUNT(*)
    FROM typed_events
    WHERE price IS NULL
       OR NOT isfinite(price)
       OR price < 0
""").fetchone()[0]

missing_rows = []

for column in EXPECTED_COLUMNS:
    count = con.execute(f"""
        SELECT COUNT(*)
        FROM normalized_events
        WHERE {column} IS NULL
    """).fetchone()[0]

    missing_rows.append({
        "column": column,
        "missing_count": count,
        "missing_percentage": round(
            100 * count / source_count, 4
        ) if source_count else 0
    })

pd.DataFrame(missing_rows).to_csv(
    REPORT_DIR / "missing_values_report.csv",
    index=False
)

quality_report = {
    "raw_rows": raw_count,
    "exact_duplicate_check_enabled": CHECK_EXACT_DUPLICATES,
    "exact_duplicate_rows_found": exact_duplicates_found,
    "exact_duplicate_removal_enabled": REMOVE_EXACT_DUPLICATES,
    "exact_duplicate_rows_removed": duplicates_removed,
    "rows_after_duplicate_handling": source_count,
    "cleaned_rows": clean_count,
    "excluded_rows": source_count - clean_count,
    "invalid_or_missing_timestamps": invalid_timestamp_count,
    "invalid_or_missing_event_types": invalid_event_type_count,
    "invalid_or_missing_prices": invalid_price_count,
}

with open(
    REPORT_DIR / "data_quality_report.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(quality_report, file, indent=4)

print("\nData quality report:")
print(json.dumps(quality_report, indent=4))


# ============================================================
# 9. EXPORT CLEANED DATA TO PARQUET
# ============================================================

clean_path = sql_string(CLEAN_DIR.as_posix())

con.execute(f"""
    COPY (
        SELECT * FROM cleaned_events
    )
    TO {clean_path}
    (
        FORMAT PARQUET,
        PARTITION_BY (event_date),
        COMPRESSION ZSTD
    )
""")

print(f"\nCleaned data saved to: {CLEAN_DIR}")


# ============================================================
# 10. READ CLEANED PARQUET FILES
# ============================================================

parquet_glob = sql_string(
    (CLEAN_DIR / "**" / "*.parquet").as_posix()
)

con.execute(f"""
    CREATE VIEW parquet_events AS
    SELECT *
    FROM read_parquet(
        {parquet_glob},
        hive_partitioning = true
    )
""")


# ============================================================
# 11. BUILD PRODUCT-SESSION FUNNEL
# ============================================================

# Funnel unit: user + session + product.
#
# View: product was viewed.
# Cart: a cart event occurred after the first view.
# Purchase: a purchase occurred after the qualifying cart.
#
# Missing user/session/product IDs are excluded from the funnel.
# remove_from_cart is retained in cleaned events but is not
# used to determine whether the cart was continuously retained.

con.execute("""
    CREATE VIEW product_session_funnel AS

    WITH valid_events AS (
        SELECT *
        FROM parquet_events
        WHERE user_id IS NOT NULL
          AND user_session IS NOT NULL
          AND product_id IS NOT NULL
    ),

    first_views AS (
        SELECT
            user_id,
            user_session,
            product_id,
            MIN(event_time) AS first_view
        FROM valid_events
        WHERE event_type = 'view'
        GROUP BY user_id, user_session, product_id
    ),

    first_carts AS (
        SELECT
            v.user_id,
            v.user_session,
            v.product_id,
            v.first_view,
            MIN(e.event_time) AS qualifying_cart
        FROM first_views v
        LEFT JOIN valid_events e
          ON e.user_id = v.user_id
         AND e.user_session = v.user_session
         AND e.product_id = v.product_id
         AND e.event_type = 'cart'
         AND e.event_time > v.first_view
        GROUP BY
            v.user_id, v.user_session, v.product_id, v.first_view
    ),

    first_purchases AS (
        SELECT
            c.user_id,
            c.user_session,
            c.product_id,
            c.first_view,
            c.qualifying_cart,
            MIN(e.event_time) AS qualifying_purchase
        FROM first_carts c
        LEFT JOIN valid_events e
          ON e.user_id = c.user_id
         AND e.user_session = c.user_session
         AND e.product_id = c.product_id
         AND e.event_type = 'purchase'
         AND c.qualifying_cart IS NOT NULL
         AND e.event_time > c.qualifying_cart
        GROUP BY
            c.user_id,
            c.user_session,
            c.product_id,
            c.first_view,
            c.qualifying_cart
    ),

    dimensions AS (
        SELECT
            user_id,
            user_session,
            product_id,
            ANY_VALUE(category_code) AS category_code,
            ANY_VALUE(brand) AS brand
        FROM valid_events
        GROUP BY user_id, user_session, product_id
    )

    SELECT
        p.user_id,
        p.user_session,
        p.product_id,
        p.first_view,
        p.qualifying_cart,
        p.qualifying_purchase,
        d.category_code,
        d.brand,
        1 AS viewed,

        CASE
            WHEN p.qualifying_cart IS NOT NULL THEN 1
            ELSE 0
        END AS added_to_cart,

        CASE
            WHEN p.qualifying_purchase IS NOT NULL THEN 1
            ELSE 0
        END AS purchased

    FROM first_purchases p
    LEFT JOIN dimensions d
      ON d.user_id = p.user_id
     AND d.user_session = p.user_session
     AND d.product_id = p.product_id
""")


# ============================================================
# 12. EXPORT FUNNEL REPORTS
# ============================================================

overall = export_funnel_summary(
    [],
    "overall_funnel.csv"
)

daily = export_funnel_summary(
    [("CAST(first_view AS DATE)", "view_date")],
    "daily_funnel.csv"
)

category = export_funnel_summary(
    [("COALESCE(category_code, 'Unknown')", "category")],
    "category_funnel.csv"
)

brand = export_funnel_summary(
    [("COALESCE(brand, 'Unknown')", "brand")],
    "brand_funnel.csv"
)


# ============================================================
# 13. EXPORT PRODUCT-SESSION FUNNEL
# ============================================================

funnel_path = sql_string(
    (OUTPUT_DIR / "product_session_funnel.parquet").as_posix()
)

con.execute(f"""
    COPY product_session_funnel
    TO {funnel_path}
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("\nOverall funnel:")
print(overall.to_string(index=False))

print("\nReports saved to:", REPORT_DIR)


# ============================================================
# 14. CLOSE DATABASE
# ============================================================

con.close()

print("\nAnalysis completed successfully.")
print("Output folder:", OUTPUT_DIR)

E-COMMERCE DATA CLEANING AND FUNNEL ANALYSIS
Input file: C:\Users\Kavya m\OneDrive\Desktop\tsts3\2019-Nov.csv
Output folder: C:\Users\Kavya m\OneDrive\Desktop\tsts3\ecommerce_analysis_20260928_144728


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Raw rows: 67,501,979

Checking exact duplicate rows...
This may take considerable time and disk space.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exact duplicate rows found: 100,519


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Data quality report:
{
    "raw_rows": 67501979,
    "exact_duplicate_check_enabled": true,
    "exact_duplicate_rows_found": 100519,
    "exact_duplicate_removal_enabled": true,
    "exact_duplicate_rows_removed": 100519,
    "rows_after_duplicate_handling": 67401460,
    "cleaned_rows": 67401460,
    "excluded_rows": 0,
    "invalid_or_missing_timestamps": 0,
    "invalid_or_missing_event_types": 0,
    "invalid_or_missing_prices": 0
}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Cleaned data saved to: C:\Users\Kavya m\OneDrive\Desktop\tsts3\ecommerce_analysis_20260928_144728\cleaned_parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))